In [2]:
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [3]:
angry_train = np.load('../hybrid/data_train/angry_hybrid_features.npy')
disgust_train = np.load('../hybrid/data_train/disgust_hybrid_features.npy')
fear_train = np.load('../hybrid/data_train/fear_hybrid_features.npy') 
happy_train = np.load('../hybrid/data_train/happy_hybrid_features.npy')
neutral_train = np.load('../hybrid/data_train/neutral_hybrid_features.npy')
sad_train = np.load('../hybrid/data_train/sad_hybrid_features.npy')
suprise_train = np.load('../hybrid/data_train/surprise_hybrid_features.npy')

In [4]:
angry_test = np.load('../hybrid/data_test/angry_hybrid_features.npy')
disgust_test = np.load('../hybrid/data_test/disgust_hybrid_features.npy')
fear_test = np.load('../hybrid/data_test/fear_hybrid_features.npy') 
happy_test = np.load('../hybrid/data_test/happy_hybrid_features.npy')
neutral_test = np.load('../hybrid/data_test/neutral_hybrid_features.npy')
sad_test = np.load('../hybrid/data_test/sad_hybrid_features.npy')
suprise_test = np.load('../hybrid/data_test/surprise_hybrid_features.npy')

In [5]:
train_data = {
    'angry': angry_train,
    'disgust': disgust_train,
    'fear': fear_train,
    'happy': happy_train,
    'neutral': neutral_train,
    'sad': sad_train,
    'suprise': suprise_train
}

In [6]:
test_data = {
    'angry': angry_test,
    'disgust': disgust_test,
    'fear': fear_test,
    'happy': happy_test,
    'neutral': neutral_test,
    'sad': sad_test,
    'suprise': suprise_test
}

In [7]:
train_X_list = []
train_y_list = []

for emotion, data in train_data.items():
    # Reshape each sample from (90, 2) to a flat vector of length 180
    train_X_list.append(data.reshape(data.shape[0], -1))
    # Create an array of labels for the emotion
    train_y_list.append(np.full((data.shape[0],), emotion))


X_train = np.concatenate(train_X_list, axis=0)
y_train = np.concatenate(train_y_list, axis=0)

In [8]:
test_X_list = []
test_y_list = []

for emotion, data in test_data.items():
    test_X_list.append(data.reshape(data.shape[0], -1))
    test_y_list.append(np.full((data.shape[0],), emotion))

X_test = np.concatenate(test_X_list, axis=0)
y_test = np.concatenate(test_y_list, axis=0)

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [10]:
import numpy as np

class CustomSVM:
    def __init__(self, C=1.0, gamma=0.1, max_iter=1000, lr=0.001):
        self.C = C                    # Regularization parameter
        self.gamma = gamma            # RBF kernel gamma
        self.max_iter = max_iter      # Number of training iterations
        self.lr = lr                  # Learning rate

    def rbf_kernel(self, X1, X2):
        # Compute the RBF (Gaussian) kernel matrix
        X1_sq = np.sum(X1**2, axis=1).reshape(-1, 1)
        X2_sq = np.sum(X2**2, axis=1).reshape(1, -1)
        dist_sq = X1_sq + X2_sq - 2 * np.dot(X1, X2.T)
        return np.exp(-self.gamma * dist_sq)

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.models = {}

        for cls in self.classes:
            # Create binary labels for one-vs-rest
            binary_y = np.where(y == cls, 1, -1)
            n_samples = X.shape[0]
            alpha = np.zeros(n_samples)
            K = self.rbf_kernel(X, X)

            # Gradient descent to optimize alpha
            for _ in range(self.max_iter):
                for i in range(n_samples):
                    margin = np.dot(alpha * binary_y, K[:, i])
                    grad = 1 - binary_y[i] * margin
                    alpha[i] += self.lr * grad
                    alpha[i] = min(max(alpha[i], 0), self.C)

            # Store support vectors and parameters
            sv = alpha > 1e-5
            self.models[cls] = {
                'X': X[sv],
                'y': binary_y[sv],
                'alpha': alpha[sv]
            }

    def project(self, X, model):
        # Project test samples onto the decision boundary using the RBF kernel
        K = self.rbf_kernel(X, model['X'])
        return np.dot(K, model['alpha'] * model['y'])

    def predict(self, X):
        # Compute decision values for each class and take the class with the highest value
        scores = np.zeros((X.shape[0], len(self.classes)))
        for idx, cls in enumerate(self.classes):
            scores[:, idx] = self.project(X, self.models[cls])
        return self.classes[np.argmax(scores, axis=1)]


In [11]:
svm = CustomSVM(C=1.0, gamma=0.05, max_iter=500, lr=0.001)
svm.fit(X_train_scaled, y_train)

y_pred = svm.predict(X_test_scaled)
accuracy = np.mean(y_pred == y_test)
print("\nTest Accuracy (7-class):", accuracy)



Test Accuracy (7-class): 0.5152291917973462


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Predict with your custom SVM
y_pred = svm.predict(X_test_scaled)

# Compute evaluation metrics
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Optional: macro and weighted averages
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')

print("\nMacro F1-score:", f1_macro)
print("Weighted F1-score:", f1_weighted)
print("Macro Precision:", precision_macro)
print("Macro Recall:", recall_macro)


Confusion Matrix:
[[ 238    1   66  167  200  119   44]
 [  19   25    7   10   13   19    4]
 [  84    0  215  123  232  179  100]
 [  45    0   39 1402  114   86   27]
 [  69    0   83  133  634  210   52]
 [ 106    0  101  157  284  394   52]
 [  32    2   54   61   81   40  509]]

Classification Report:
              precision    recall  f1-score   support

       angry     0.4013    0.2850    0.3333       835
     disgust     0.8929    0.2577    0.4000        97
        fear     0.3805    0.2304    0.2870       933
       happy     0.6829    0.8184    0.7446      1713
     neutral     0.4069    0.5368    0.4629      1181
         sad     0.3763    0.3601    0.3681      1094
     suprise     0.6459    0.6534    0.6496       779

    accuracy                         0.5152      6632
   macro avg     0.5410    0.4489    0.4637      6632
weighted avg     0.5039    0.5152    0.5000      6632


Macro F1-score: 0.4636547562682657
Weighted F1-score: 0.49997541035115955
Macro Precision: 0.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

# Predict with your custom SVM
y_pred = svm.predict(X_test_scaled)

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=svm.classes,   # <- fixed here
    columns=svm.classes  # <- and here
)
print("\nConfusion Matrix:")
print(conf_matrix_df)

# Classification Report
report = classification_report(y_test, y_pred, digits=4, output_dict=True)
report_df = pd.DataFrame(report).transpose()
print("\nClassification Report:")
print(report_df.round(4))

# Overall Metrics
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')

overall_metrics = pd.DataFrame({
    "Metric": ["Macro F1-score", "Weighted F1-score", "Macro Precision", "Macro Recall"],
    "Score": [f1_macro, f1_weighted, precision_macro, recall_macro]
})

print("\nOverall Metrics:")
print(overall_metrics.set_index("Metric").round(4))



📊 Confusion Matrix:
         angry  disgust  fear  happy  neutral  sad  suprise
angry      238        1    66    167      200  119       44
disgust     19       25     7     10       13   19        4
fear        84        0   215    123      232  179      100
happy       45        0    39   1402      114   86       27
neutral     69        0    83    133      634  210       52
sad        106        0   101    157      284  394       52
suprise     32        2    54     61       81   40      509

🧾 Classification Report:
              precision  recall  f1-score    support
angry            0.4013  0.2850    0.3333   835.0000
disgust          0.8929  0.2577    0.4000    97.0000
fear             0.3805  0.2304    0.2870   933.0000
happy            0.6829  0.8184    0.7446  1713.0000
neutral          0.4069  0.5368    0.4629  1181.0000
sad              0.3763  0.3601    0.3681  1094.0000
suprise          0.6459  0.6534    0.6496   779.0000
accuracy         0.5152  0.5152    0.5152     0.5